In [70]:
import jax
import jax.numpy as jnp

def lossless_contact(non_dim_eta):
    return jnp.abs(non_dim_eta)**2

def loss_contact(scattering_length, eta):
    a_s = jnp.asarray(scattering_length, dtype=jnp.complex64) 
    return jnp.imag(1/a_s) * jnp.abs(eta)**2
    
def diff(grid):
    return grid[1:]-grid[:-1]

def cumulative_trapez_int(integrand, dx):
    interval_integrals = 0.5 * dx * (integrand[:-1] + integrand[1:])
    initial_value = jnp.zeros((1,), dtype=interval_integrals.dtype)
    return jnp.concatenate((initial_value, jnp.cumsum(interval_integrals)))
    
def trapez_int(integrand, dx):
    integral = 0.5 * dx * (integrand[:-1] + integrand[1:])
    return jnp.sum(integral)
    
def mol_density(scattering_length, eta, dt):
    # for inelastic models with complex scattering lengths.
    a_s = jnp.asarray(scattering_length, dtype=jnp.complex64) 
    eta = jnp.asarray(eta, dtype=jnp.complex64) 

    prefix = (1 / (2 * jnp.pi))
    contact = loss_contact(a_s, eta)
    mol_den = prefix * trapez_int(contact, dt)
    return mol_den 
    
def energy_density(scattering_length, eta):
    # for elastic models with real scattering lengths. 
    a_s = jnp.asarray(scattering_length, dtype=jnp.float32) 
    eta = jnp.asarray(eta, dtype=jnp.complex64) 
    
    prefix = 1/(8*jnp.pi)

    integr = lossless_contact(eta) / (a_s**2)
    ener_den = prefix * trapez_int(integr, diff(a_s))

    return ener_den
    
def grid_mol_density(scattering_length, eta, dt):
    # for inelastic models with complex scattering lengths.
    a_s = jnp.asarray(scattering_length, dtype=jnp.complex64) 
    eta = jnp.asarray(eta, dtype=jnp.complex64) 

    prefix = (1 / (2 * jnp.pi))
    contact = loss_contact(a_s, eta)
    mol_den = prefix * cumulative_trapez_int(contact, dt)
    return mol_den 

def grid_energy_density(scattering_length, eta):
    # for elastic models with real scattering lengths. 
    a_s = jnp.asarray(scattering_length, dtype=jnp.float32) 
    eta = jnp.asarray(eta, dtype=jnp.complex64) 
    
    prefix = 1/(8*jnp.pi)

    integr = lossless_contact(eta) / (a_s**2)
    ener_den = prefix * cumulative_trapez_int(integr, diff(a_s))

    return ener_den
    

    
